In [1]:
list.of.packages <- c("ggplot2", "Rcpp", "grf", "caret", "mltools", "rpart", "minpack.lm", "doParallel", "rattle", "anytime","rlist")
list.of.packages <- c(list.of.packages, "zoo", "dtw", "foreach", "evaluate","rlist","data.table","plyr","dplyr", "fs")


new.packages <- list.of.packages[!(list.of.packages %in% installed.packages()[,"Package"])]
if(length(new.packages)) install.packages(new.packages, lib='/home/zwang937/local/R_libs', repos="http://cran.us.r-project.org", dependencies = TRUE, INSTALL_opts = '--no-lock')


lapply(list.of.packages, require, character.only = TRUE)

# Register the cluster
registerDoParallel(cores=detectCores())

Loading required package: ggplot2

Loading required package: Rcpp

Loading required package: grf

Loading required package: caret

Loading required package: lattice

Loading required package: mltools

Loading required package: rpart

Loading required package: minpack.lm

Loading required package: doParallel

Loading required package: foreach

Loading required package: iterators

Loading required package: parallel

Loading required package: rattle

Loading required package: tibble

Loading required package: bitops

Rattle: A free graphical interface for data science with R.
Version 5.5.1 Copyright (c) 2006-2021 Togaware Pty Ltd.
Type 'rattle()' to shake, rattle, and roll your data.

Loading required package: anytime

Loading required package: rlist

Loading required package: zoo


Attaching package: ‘zoo’


The following objects are masked from ‘package:base’:

    as.Date, as.Date.numeric


Loading required package: dtw

Loading required package: proxy


Attaching package: ‘proxy’


Th

[[1]]
[1] TRUE

[[2]]
[1] TRUE

[[3]]
[1] TRUE

[[4]]
[1] TRUE

[[5]]
[1] TRUE

[[6]]
[1] TRUE

[[7]]
[1] TRUE

[[8]]
[1] TRUE

[[9]]
[1] TRUE

[[10]]
[1] TRUE

[[11]]
[1] TRUE

[[12]]
[1] TRUE

[[13]]
[1] TRUE

[[14]]
[1] TRUE

[[15]]
[1] TRUE

[[16]]
[1] TRUE

[[17]]
[1] TRUE

[[18]]
[1] TRUE

[[19]]
[1] TRUE

[[20]]
[1] TRUE

In [17]:
dfs_depth <- function(tree_nodes_list, node_index = 1, current_depth = 1) {
  # Base case: if the node is a leaf, return the current depth
  if (tree_nodes_list[[node_index]]$is_leaf) {
    return(current_depth)
  }
  
  # Recursive case: calculate depth of the child nodes
  left_child_index <- tree_nodes_list[[node_index]]$left_child
  right_child_index <- tree_nodes_list[[node_index]]$right_child
  
  left_depth <- dfs_depth(tree_nodes_list, left_child_index, current_depth + 1)
  right_depth <- dfs_depth(tree_nodes_list, right_child_index, current_depth + 1)
  
  # Return the maximum depth among the child nodes
  return(max(left_depth, right_depth))
}

forest_depths <- function(grf_forest){
    forest_depths_vector <- foreach(i = 1:grf_forest$`_num_trees`) %dopar%{
        test_tree <- (grf::get_tree(grf_forest,i))
        test_tree_nodes <- test_tree$nodes
        dfs_depth(test_tree_nodes)
    }
    return(forest_depths_vector)
}

In [4]:
print("Setting up directory path")
Individual_RDS_directory <- "../../../data/output/individual_county_grf_windowsize=2_numtrees=100"

# Get a list of all subfolder names (integer names)
subfolder_names <- dir_ls(Individual_RDS_directory, type = "dir", recurse = FALSE, full_path = FALSE)

# Create an empty list to store the loaded objects
#loaded_objects <- list()
print("Loading Objects")
loaded_objects <- foreach(subfolder_name = subfolder_names, .errorhandling="pass") %dopar% {
  # Get the full path to the subfolder
  directory_name <- basename(strsplit(subfolder_name, "/")[[1]])
  fips_string <- directory_name[length(directory_name)]
  # Get the file names of the RDS objects in the subfolder
  #subfolder_feature_importance <- list()
  loaded_objects_sub <- foreach(cutoff =seq(100, 1000, by = 100), .errorhandling="pass") %do% {
      fname <- paste0("grf_individual_county_fips=",fips_string,"_cutoff=",toString(cutoff),".rds")
      print(fname)
      rds_file_path <- file.path(subfolder_name, fname)
      # Try reading the RDS file and extract depth
      tryCatch({
        loaded_object <- readRDS(rds_file_path)
        #depths <- forest_depths(loaded_object)
        #depths
        loaded_object
      }, error = function(e) {
        # Return NULL in case of error
        NULL
      })
  }
  #print(subfolder_feature_importance)
  # Return the loaded objects for this subfolder
  #loaded_objects[[subfolder_name]] <- subfolder_feature_importance
  compact(loaded_objects_sub)
}
print("Done!")

[1] "Setting up directory path"
[1] "Loading Objects"


In [5]:
loaded_objects <- loaded_objects[lengths(loaded_objects) > 0]


In [8]:
unlisted_loaded_objects<-unlist(loaded_objects, recursive = FALSE)

In [9]:
length(unlisted_loaded_objects)

[1] 28501

In [ ]:
#unlisted_loaded_objects[lengths(unlisted_loaded_objects) != 467]

In [11]:
(unlisted_loaded_objects[[1]])

GRF forest object of type causal_forest 
Number of trees: 100 
Number of training samples: 50 
Variable importance: 
  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19  20 
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
 21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38  39  40 
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
 41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56  57  58  59  60 
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
 61  62  63  64  65  66  67  68  69  70  71  72  73  74  75  76  77  78  79  80 
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
 81  82  83  84  85  86  87  88  89  90  91  92  93  94  95  96  97  98  99 100 
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 1

### Loop through list of GRF Objects

In [21]:
test_object <- unlisted_loaded_objects[[1]]
depths_matrix <- matrix(NA, nrow=length(unlisted_loaded_objects), ncol = test_object$`_num_trees`)
for (i in 1:length(unlisted_loaded_objects)) {
    print(i)
    grf_object <- unlisted_loaded_objects[[i]]
    depths_vector <- forest_depths(grf_object)
    depths_matrix[i,] <- as.vector(depths_vector)
}


[1] 1
[1] 2


ERROR: Error in depths_matrix[i, ] <- as.vector(depths_vector): incorrect number of subscripts on matrix


In [26]:
length(unlisted_loaded_objects)

[1] 28501

In [28]:
x <- as.vector(forest_depths(grf_object))

In [29]:
depths_matrix[1,] <- x

In [32]:
fwrite(x, "test_depths.csv")

In [ ]:
result_matrix <- matrix(NA, nrow = length(unlisted_loaded_objects), ncol = lengths(unlisted_loaded_objects)[1])
for (i in 1:length(unlisted_loaded_objects)) {
  result_matrix[i, ] <- as.vector(unlisted_loaded_objects[[i]])
}

In [ ]:
colnames(result_matrix) <- colnames(unlisted_loaded_objects[[1]])

In [ ]:
my_datatable <- data.table(result_matrix)
my_datatable

In [ ]:
fwrite(my_datatable, "individual_grf_feature_importance.csv", row.names=FALSE)

In [ ]:
test_fpath <-file.path(subfolder_names[[2]], paste0("grf_individual_county_fips=","10001","_cutoff=",toString(1000),".rds"))
print(test_fpath)
test_grf<-readRDS(test_fpath)

In [ ]:
test_grf$`_child_nodes`

In [ ]:
test_tree <- (grf::get_tree(test_grf,1))

In [ ]:
(test_tree$nodes)[[1]]

In [ ]:
dfs_depth <- function(tree_nodes_list, node_index = 1, current_depth = 1) {
  # Base case: if the node is a leaf, return the current depth
  if (tree_nodes_list[[node_index]]$is_leaf) {
    return(current_depth)
  }
  
  # Recursive case: calculate depth of the child nodes
  left_child_index <- tree_nodes_list[[node_index]]$left_child
  right_child_index <- tree_nodes_list[[node_index]]$right_child
  
  left_depth <- dfs_depth(tree_nodes_list, left_child_index, current_depth + 1)
  right_depth <- dfs_depth(tree_nodes_list, right_child_index, current_depth + 1)
  
  # Return the maximum depth among the child nodes
  return(max(left_depth, right_depth))
}

forest_depths <- function(grf_forest){
    forest_depths_vector <- foreach(i = 1:grf_forest$`_num_trees`) %dopar%{
        test_tree <- (grf::get_tree(grf_forest,i))
        test_tree_nodes <- test_tree$nodes
        dfs_depth(test_tree_nodes)
    }
    return(forest_depths_vector)
}

In [ ]:
forest_depths(test_grf)

In [ ]:
grf::get_tree(test_grf,6)

In [ ]:
### Use get_leaf to see for each day, county, in which leaf does the datablock fall into
### return the size of that leaf, to tell us how many other blocks also fall in there
### large leaf => Long memory in this case
### get_leaf_node(tree, newdata, node.id = TRUE)

In [ ]:
break

In [ ]:

print("Setting up directory path")
Individual_RDS_directory <- "../../../data/output/individual_county_grf_windowsize=2_numtrees=100"

# Get a list of all subfolder names (integer names)
subfolder_names <- dir_ls(Individual_RDS_directory, type = "dir", recurse = FALSE, full_path = FALSE)

# Create an empty list to store the loaded objects
#loaded_objects <- list()
print("Loading Objects")
loaded_objects <- foreach(subfolder_name = subfolder_names, .errorhandling="pass") %dopar% {
  # Get the full path to the subfolder
  directory_name <- basename(strsplit(subfolder_name, "/")[[1]])
  fips_string <- directory_name[length(directory_name)]
  # Get the file names of the RDS objects in the subfolder
  #subfolder_feature_importance <- list()
  subfolder_feature_importance <- foreach(cutoff =seq(100, 1000, by = 100), .errorhandling="pass") %do% {
      fname <- paste0("grf_individual_county_fips=",fips_string,"_cutoff=",toString(cutoff),".rds")
      #print(fname)
      rds_file_path <- file.path(subfolder_name, fname)
      # Try reading the RDS file and extract feature importance
      tryCatch({
        loaded_object <- readRDS(rds_file_path)
        feature_importance <- t(grf::variable_importance(loaded_object))
        feature_names <- colnames(loaded_object$X.orig)
        colnames(feature_importance) <- feature_names
        feature_importance
      }, error = function(e) {
        # Return NULL in case of error
        NULL
      })
  }
  #print(subfolder_feature_importance)
  # Return the loaded objects for this subfolder
  #loaded_objects[[subfolder_name]] <- subfolder_feature_importance
  compact(subfolder_feature_importance)
}